# SQL UDFs & Higher-Order Functions

Two powerful tools extend what you can express in pure SQL: **SQL UDFs** let you encapsulate reusable logic as named functions registered in the metastore; **higher-order functions** let you operate on arrays and maps inline, without exploding them into rows first.

In this notebook we'll cover:
- Creating, using, and dropping SQL UDFs
- UDF scoping — temp vs persistent, Unity Catalog namespace
- UDF limitations (Catalyst visibility)
- Higher-order functions: `TRANSFORM`, `FILTER`, `EXISTS`, `FORALL`, `REDUCE`
- Array functions: `EXPLODE`, `COLLECT_LIST`, `ARRAY_CONTAINS`, `FLATTEN`, and more
- Map functions: `MAP_KEYS`, `MAP_VALUES`, `EXPLODE` on maps

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS udf_demo")
spark.sql("USE SCHEMA udf_demo")

## SQL UDFs

### Creating a SQL UDF

```sql
CREATE OR REPLACE FUNCTION schema.function_name(param1 TYPE, param2 TYPE, ...)
RETURNS return_type
RETURN expression;
```

The body is a single SQL expression — any built-in function, operator, or `CASE WHEN` can appear there. The function is registered in the metastore and persists across sessions.

**Example — classify order amount:**

```sql
CREATE OR REPLACE FUNCTION udf_demo.order_tier(amount DOUBLE)
RETURNS STRING
RETURN CASE
  WHEN amount >= 500 THEN 'high'
  WHEN amount >= 100 THEN 'medium'
  ELSE                    'low'
END;
```

Once created, use it anywhere a built-in function would appear:

```sql
SELECT order_id, amount, udf_demo.order_tier(amount) AS tier
FROM   silver.orders;
```

### Temporary SQL UDFs

Drop the schema qualifier and add `TEMPORARY` to create a session-scoped function:

```sql
CREATE OR REPLACE TEMPORARY FUNCTION order_tier(amount DOUBLE)
RETURNS STRING
RETURN CASE WHEN amount >= 500 THEN 'high' ELSE 'low' END;
```

Temp UDFs live only for the current SparkSession — they are not persisted in the metastore.

### UDFs Calling Other UDFs

A SQL UDF body can call other SQL UDFs — useful for building composable logic:

```sql
CREATE OR REPLACE FUNCTION udf_demo.is_high_value(amount DOUBLE)
RETURNS BOOLEAN
RETURN udf_demo.order_tier(amount) = 'high';
```

### Dropping UDFs

```sql
DROP FUNCTION IF EXISTS udf_demo.order_tier;
DROP TEMPORARY FUNCTION IF EXISTS order_tier;
```

In [ ]:
# Create a persistent SQL UDF
spark.sql("""
  CREATE OR REPLACE FUNCTION udf_demo.mask_email(email STRING)
  RETURNS STRING
  RETURN CONCAT(
    LEFT(email, 2),
    REPEAT('*', LENGTH(SPLIT(email, '@')[0]) - 2),
    '@',
    SPLIT(email, '@')[1]
  )
""")

# Use it in a query
spark.sql("""
  SELECT
    email,
    udf_demo.mask_email(email) AS masked_email
  FROM VALUES
    ('alice@example.com'),
    ('bob@databricks.com'),
    ('carol.smith@company.co.uk')
  AS t(email)
""").show(truncate=False)

In [ ]:
# UDF calling another UDF
spark.sql("""
  CREATE OR REPLACE FUNCTION udf_demo.order_tier(amount DOUBLE)
  RETURNS STRING
  RETURN CASE
    WHEN amount >= 500 THEN 'high'
    WHEN amount >= 100 THEN 'medium'
    ELSE                    'low'
  END
""")

spark.sql("""
  CREATE OR REPLACE FUNCTION udf_demo.is_high_value(amount DOUBLE)
  RETURNS BOOLEAN
  RETURN udf_demo.order_tier(amount) = 'high'
""")

spark.sql("""
  SELECT amount, udf_demo.order_tier(amount) AS tier,
                 udf_demo.is_high_value(amount) AS is_high
  FROM VALUES (50.0), (150.0), (600.0) AS t(amount)
""").show()

In [ ]:
# Inspect a UDF — DESCRIBE FUNCTION shows definition
spark.sql("DESCRIBE FUNCTION EXTENDED udf_demo.mask_email").show(truncate=False)

## UDF Limitations

SQL UDFs are **opaque to the Catalyst optimizer**. Catalyst cannot look inside a UDF body to push filters, reorder operations, or apply other optimizations — it treats the UDF as a black box.

**Implication:** If you can express the same logic with built-in functions, prefer built-ins — Catalyst can optimize them. Use UDFs for logic that genuinely cannot be expressed with built-ins, or for readability/reuse.

| | Built-in function | SQL UDF |
|---|---|---|
| Catalyst optimisation | Full | None (black box) |
| Serialization overhead | None | None (SQL UDFs stay in JVM) |
| Metastore persistence | No | Yes |
| Reusable across queries | No | Yes |

> Python UDFs (covered in notebook 08) have an additional overhead: they require serializing data between the JVM and the Python process. SQL UDFs avoid this — their body runs on the JVM — so they're significantly faster than Python UDFs when both options exist.

## Higher-Order Functions

Higher-order functions take a **lambda expression** as an argument and apply it to each element of an array (or each key/value of a map). The lambda syntax in Spark SQL is:

```
element -> expression
(element, index) -> expression    (for functions that expose index)
(acc, element) -> expression      (for REDUCE)
```

### TRANSFORM — Apply a Function to Every Element

Returns a new array of the same size with each element replaced by the result of the lambda.

```sql
-- Double every price in an array
TRANSFORM(prices, p -> p * 2)

-- Uppercase every tag
TRANSFORM(tags, t -> UPPER(t))

-- Apply a UDF to every element
TRANSFORM(amounts, a -> udf_demo.order_tier(a))
```

### FILTER — Keep Elements Matching a Condition

Returns a new array containing only elements for which the lambda returns `true`.

```sql
-- Keep only prices above 100
FILTER(prices, p -> p > 100)

-- Keep only non-null tags
FILTER(tags, t -> t IS NOT NULL)
```

### EXISTS — Does Any Element Match?

Returns `true` if at least one element satisfies the lambda.

```sql
-- Does this order have any item over 500?
EXISTS(items, i -> i.price > 500)
```

### FORALL — Do All Elements Match?

Returns `true` only if every element satisfies the lambda.

```sql
-- Are all items in stock?
FORALL(items, i -> i.in_stock = true)
```

### REDUCE — Aggregate an Array to a Scalar

Accumulates a result across all elements. Takes a zero value, a merge lambda, and an optional finish lambda.

```sql
-- Sum all prices in an array
REDUCE(prices, 0D, (acc, p) -> acc + p)

-- Sum then apply a discount
REDUCE(prices, 0D, (acc, p) -> acc + p, total -> total * 0.9)
```

In [ ]:
# Higher-order functions in action
spark.sql("""
  SELECT
    order_id,
    item_prices,
    TRANSFORM(item_prices, p -> ROUND(p * 1.2, 2))  AS prices_with_vat,
    FILTER(item_prices,  p -> p > 50)               AS expensive_items,
    EXISTS(item_prices,  p -> p > 200)              AS has_premium_item,
    FORALL(item_prices,  p -> p > 0)                AS all_positive,
    REDUCE(item_prices, 0D, (acc, p) -> acc + p)    AS order_total
  FROM VALUES
    ('ORD001', ARRAY(25.0, 80.0, 250.0)),
    ('ORD002', ARRAY(15.0, 40.0, 30.0)),
    ('ORD003', ARRAY(500.0, 120.0))
  AS t(order_id, item_prices)
""").show(truncate=False)

## Array Functions

### EXPLODE — Array to Rows

Transforms one row with an array column into multiple rows — one per element. The original row's other columns are duplicated.

```sql
SELECT order_id, EXPLODE(tags) AS tag FROM orders;
```

| Variant | Behaviour |
|---------|----------|
| `EXPLODE(array)` | Drops rows where array is null or empty |
| `EXPLODE_OUTER(array)` | Keeps rows where array is null — element is null |
| `POSEXPLODE(array)` | Returns `(pos, col)` — position index + element |

### COLLECT_LIST / COLLECT_SET — Rows to Array

The inverse of EXPLODE — aggregate multiple rows into a single array.

```sql
-- All products per order
SELECT order_id, COLLECT_LIST(product_id) AS products
FROM   order_items
GROUP BY order_id;

-- Distinct products only
SELECT order_id, COLLECT_SET(product_id) AS unique_products
FROM   order_items
GROUP BY order_id;
```

### Other Commonly Tested Array Functions

```sql
SIZE(array)                           -- number of elements (null-safe: returns -1 for null)
ARRAY_CONTAINS(array, value)          -- boolean membership test
ARRAY_DISTINCT(array)                 -- remove duplicates
ARRAY_SORT(array)                     -- ascending sort
ARRAY_UNION(a1, a2)                   -- union of two arrays (distinct)
ARRAY_INTERSECT(a1, a2)               -- elements in both
ARRAY_EXCEPT(a1, a2)                  -- elements in a1 but not a2
FLATTEN(array_of_arrays)              -- one level of nesting removed
ZIP_WITH(a1, a2, (x, y) -> x + y)    -- element-wise combine
ARRAYS_ZIP(a1, a2)                    -- zip into array of structs
SLICE(array, start, length)           -- sub-array
ARRAY_POSITION(array, value)          -- 1-based index of first match
```

In [ ]:
# EXPLODE — one row per tag
spark.sql("""
  SELECT order_id, EXPLODE(tags) AS tag
  FROM VALUES
    ('ORD001', ARRAY('electronics', 'sale', 'premium')),
    ('ORD002', ARRAY('clothing')),
    ('ORD003', NULL)
  AS t(order_id, tags)
""").show()

In [ ]:
# EXPLODE_OUTER — keeps null arrays
spark.sql("""
  SELECT order_id, EXPLODE_OUTER(tags) AS tag
  FROM VALUES
    ('ORD001', ARRAY('electronics', 'sale')),
    ('ORD003', NULL)
  AS t(order_id, tags)
""").show()

In [ ]:
# COLLECT_LIST — aggregate rows back into arrays (inverse of EXPLODE)
spark.sql("""
  SELECT customer_id,
         COLLECT_LIST(order_id)    AS all_orders,
         COLLECT_SET(region)       AS regions_visited,
         ROUND(SUM(amount), 2)     AS total_spend
  FROM VALUES
    ('CUST01', 'ORD001', 120.5, 'UK'),
    ('CUST01', 'ORD003',  89.9, 'UK'),
    ('CUST01', 'ORD007', 340.0, 'US'),
    ('CUST02', 'ORD002', 200.0, 'DE')
  AS t(customer_id, order_id, amount, region)
  GROUP BY customer_id
""").show(truncate=False)

In [ ]:
# FLATTEN + ARRAY_DISTINCT — normalise nested arrays
spark.sql("""
  SELECT
    user_id,
    all_tags,
    FLATTEN(all_tags)               AS flat_tags,
    ARRAY_DISTINCT(FLATTEN(all_tags)) AS unique_tags,
    SIZE(ARRAY_DISTINCT(FLATTEN(all_tags))) AS unique_tag_count
  FROM VALUES
    ('USR01', ARRAY(ARRAY('scala', 'java'), ARRAY('java', 'spark'))),
    ('USR02', ARRAY(ARRAY('python'), ARRAY('sql', 'python')))
  AS t(user_id, all_tags)
""").show(truncate=False)

## Map Functions

Maps are key-value collections. Spark SQL has built-in functions for creating, accessing, and exploding them.

```sql
-- Create a map literal
MAP('key1', value1, 'key2', value2)

-- Access a value by key (returns null if key absent)
my_map['key1']
ELEMENT_AT(my_map, 'key1')

-- Extract all keys or values as arrays
MAP_KEYS(my_map)     -- ARRAY of keys
MAP_VALUES(my_map)   -- ARRAY of values

-- Parse a delimited string into a map
STR_TO_MAP('a:1,b:2,c:3', ',', ':')   -- MAP('a' -> '1', 'b' -> '2', 'c' -> '3')

-- Explode map into rows of (key, value)
EXPLODE(my_map)      -- columns: key, value
```

In [ ]:
# Map creation, access, and explode
spark.sql("""
  SELECT
    order_id,
    metadata,
    metadata['source']          AS source,
    MAP_KEYS(metadata)          AS keys,
    MAP_VALUES(metadata)        AS values
  FROM VALUES
    ('ORD001', MAP('source', 'web',    'promo', 'SUMMER10')),
    ('ORD002', MAP('source', 'mobile', 'agent', 'rep_42'))
  AS t(order_id, metadata)
""").show(truncate=False)

In [ ]:
# EXPLODE a map — one row per key-value pair
spark.sql("""
  SELECT order_id, key, value
  FROM (
    SELECT order_id, EXPLODE(metadata) AS (key, value)
    FROM VALUES
      ('ORD001', MAP('source', 'web', 'promo', 'SUMMER10')),
      ('ORD002', MAP('source', 'mobile', 'agent', 'rep_42'))
    AS t(order_id, metadata)
  )
""").show()

In [ ]:
# STR_TO_MAP — parse event properties encoded as 'k:v,k:v' strings
spark.sql("""
  SELECT
    event_id,
    raw_props,
    STR_TO_MAP(raw_props, ',', ':')          AS props_map,
    STR_TO_MAP(raw_props, ',', ':')['page']  AS page
  FROM VALUES
    ('E001', 'page:checkout,user_id:CUST01,ab_group:B'),
    ('E002', 'page:home,user_id:CUST02,ab_group:A')
  AS t(event_id, raw_props)
""").show(truncate=False)

In [ ]:
# Cleanup
spark.sql("DROP FUNCTION IF EXISTS udf_demo.mask_email")
spark.sql("DROP FUNCTION IF EXISTS udf_demo.order_tier")
spark.sql("DROP FUNCTION IF EXISTS udf_demo.is_high_value")
spark.sql("DROP SCHEMA IF EXISTS udf_demo CASCADE")

## Summary

- **SQL UDFs** are created with `CREATE OR REPLACE FUNCTION schema.name(...) RETURNS type RETURN expression`. They are persisted in the metastore and reusable across sessions and queries.
- `TEMPORARY` UDFs are session-scoped — not persisted. Use them for one-off logic or testing.
- UDFs can call other UDFs, enabling composable logic.
- SQL UDFs are opaque to Catalyst — prefer built-in functions when possible. SQL UDFs are still faster than Python UDFs because they stay on the JVM.
- **Higher-order functions** operate on arrays with lambda expressions: `TRANSFORM` maps, `FILTER` filters, `EXISTS` checks any, `FORALL` checks all, `REDUCE` aggregates to a scalar.
- **`EXPLODE`** converts an array to multiple rows. **`EXPLODE_OUTER`** keeps null-array rows. **`POSEXPLODE`** also returns the element's position index.
- **`COLLECT_LIST`** / **`COLLECT_SET`** are the inverse — aggregate rows into an array. `COLLECT_SET` removes duplicates.
- **`FLATTEN`** unwraps one level of array nesting. **`ARRAY_DISTINCT`** deduplicates. **`SIZE`** returns element count.
- **Map functions**: `MAP_KEYS`, `MAP_VALUES`, bracket access `map['key']`, `EXPLODE` for key-value rows, `STR_TO_MAP` to parse delimited strings.

Next: `08-python-udfs-control-flow.ipynb` — Python UDFs, vectorized UDFs with pandas, and notebook control flow with `dbutils`.